RUN all master codes

In [1]:
# Core
import sys
import os
import numpy as np
import pandas as pd

# SQL
import sqlite3
import sqlalchemy as sa
from sqlalchemy import create_engine, text

# Spark
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pyspark
import logging

spark = SparkSession.builder \
    .appName("TanureDataScience") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .config("spark.log.level", "ERROR") \
    .config("spark.driver.extraJavaOptions", "-Dlog4j.logLevel=ERROR") \
    .config("spark.network.timeout", "120s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .getOrCreate()

logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning (GPU)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Backup (CPU only)
import tensorflow as tf

# Confirm everything is active
device = torch.device("cuda")
print(f"Environment: {sys.executable}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"SQLAlchemy: {sa.__version__}")
print(f"PySpark: {pyspark.__version__}")
print(f"Spark: {spark.version}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)} GB")
print("All systems ready including Spark!")

Environment: c:\Users\Tanur\environment\uni\tanure\Scripts\python.exe
NumPy: 2.0.2
PyTorch: 2.11.0+cu128
TensorFlow: 2.18.0
SQLAlchemy: 2.0.49
PySpark: 3.5.3
Spark: 3.5.3
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
VRAM: 23.89 GB
All systems ready including Spark!


Step 1:
Upload File and take a look to understand

In [2]:
aidi2020_df = pd.read_csv("C:/Users/Tanur/Documents/careeer path/project/downtime predictor/dataset/ai4i+2020+predictive+maintenance+dataset/ai4i2020.csv")
print(aidi2020_df.head()) #Shows the first 5 rows of the table.
print(aidi2020_df.shape) #Returns two numbers: rows and columns.
print(aidi2020_df.columns.tolist()) #Returns a list of all column names.

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

The dataset: 10,000 rows, 14 columns

Input features (what the sensors measure):

Air temperature [K] : ambient temperature around the machine
Process temperature [K] : temperature of the actual process
Rotational speed [rpm] : how fast the machine is spinning
Torque [Nm] : rotational force being applied
Tool wear [min] : how long the tool has been in use
Type : machine quality variant (L, M, H)

Target columns (what i plan to predict):

Machine failure : the main one, 1 means failure occurred
TWF : tool wear failure
HDF : heat dissipation failure
PWF : power failure
OSF : overstrain failure
RNF : random failure

Our target is Machine failure. The others are subtypes that explain why it failed, which is actually useful context for a business dashboard later.

Step 2
how many failures actually exist in the data?

In [3]:
print(aidi2020_df['Machine failure'].value_counts()) #Counts the number of occurrences of each unique value in the 'Machine failure' column.
print()
print(aidi2020_df['Machine failure'].value_counts(normalize=True) * 100) #Returns the percentage of each unique value in the 'Machine failure' column.

Machine failure
0    9661
1     339
Name: count, dtype: int64

Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


how unbalanced the dataset is, which shapes everything i do next
9,661 normal operations
339 failures
only 3.39% of records are failures

step 3
check data quality before modelling

In [4]:
print(aidi2020_df.isnull().sum()) #Checks for missing values in each column and returns the count of missing values for each column.
#.isnull() checks every single cell in the table and marks it True if it is empty, False if it has a value. 
# .sum() then counts how many Trues exist per column. all zeros meaning no missing data anywhere.
print()
print(aidi2020_df.dtypes) #Returns the data type of each column.
print()
print(aidi2020_df.describe()) #Provides summary statistics for numerical columns.

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

               UDI  Air temperature [K]  Process temperature [K]  \
count  

Step 4
prepare data for modelling

In [5]:
from sklearn.preprocessing import LabelEncoder #Importing the LabelEncoder class from the sklearn.preprocessing module, which is used to convert categorical labels into numerical format.
#For example it will turn L, M, H into 0, 1, 2 because ML models cannot process letters.

label_encoder = LabelEncoder() #Creating an instance of the LabelEncoder class.

df_clean = aidi2020_df.drop(columns=['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']) #Creates a new DataFrame called df_clean by dropping the specified columns from the original DataFrame aidi2020_df. 
#These columns are unnecessary for the analysis or modeling process.
#drop UDI and Product ID because they are just ID numbers with no predictive value.
#drop TWF, HDF, PWF, OSF, RNF because they are subtypes of failure that would leak information into the model.

le = LabelEncoder()
df_clean['Type'] = le.fit_transform(df_clean['Type']) #Applies the label encoder to the 'Type' column of the df_clean DataFrame, transforming the categorical values into numerical format.
#Creates a LabelEncoder object called le. Then it uses the fit_transform() method of the LabelEncoder to convert the 'Type' column from categorical values (like L, M, H) to numerical values (like 0, 1, 2).

x = df_clean.drop(columns=['Machine failure']) #Creates a new DataFrame x by dropping the 'Machine failure' column from df_clean. This will be used as the feature set for modeling.
y = df_clean['Machine failure'] #Creates a new Series y that contains only the 'Machine failure' column from df_clean. This will be used as the target variable for modeling.
#X are features, meaning all the input columns the model learns from
#y is the target, meaning the output column the model is trying to predict. In this case, whether the machine fails or not.

to confirm data is ready for modeling, we can check the shapes of x and y:

In [6]:
print("Features shape:", x.shape) #Prints the shape of the feature set x, which indicates the number of rows and columns in the DataFrame.
print("Target shape:", y.shape) #Prints the shape of the target variable y, which indicates the number of rows in the Series. The number of rows in x and y should be the same, confirming that each row of features corresponds to one target value.
print()
print(df_clean.head()) #Prints the first 5 rows of the cleaned DataFrame df_clean, which now contains only the relevant features and the target variable, with the 'Type' column encoded as numerical values.


Features shape: (10000, 6)
Target shape: (10000,)

   Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0     2                298.1                    308.6                    1551   
1     1                298.2                    308.7                    1408   
2     1                298.1                    308.5                    1498   
3     1                298.2                    308.6                    1433   
4     1                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Machine failure  
0         42.8                0                0  
1         46.3                3                0  
2         49.4                5                0  
3         39.5                7                0  
4         40.0                9                0  


Step 5
Split the data into training and test sets, then handle the imbalance with Synthetic Minority Over-sampling Technique (SMOTE)

In [7]:
from sklearn.model_selection import train_test_split #Importing the train_test_split function from the sklearn.model_selection module, which is used to split the dataset into training and testing sets.
from imblearn.over_sampling import SMOTE #Importing the SMOTE class from the imblearn.over_sampling module, which is used to perform Synthetic Minority Over-sampling Technique (SMOTE) to address class imbalance in the dataset.

#split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y) #Splits the feature set x and target variable y into training and testing sets. The
# The test_size=0.2 parameter indicates that 20% of the data is allocated for testing, while the remaining 80% is used for training.
# The random_state=42 parameter ensures that the split is reproducible, and stratify=y ensures that the class distribution of the target variable y is preserved in both the training and testing sets, which is important when dealing with imbalanced datasets.

print("Training set size:", x_train.shape) #Prints the shape of the training feature set x_train, which indicates the number of rows and columns in the training data.
print("Testing set size:", x_test.shape) #Prints the shape of the testing feature set x_test, which indicates the number of rows and columns in the testing data. 
#The number of rows in x_train and x_test should add up to the total number of rows in x, confirming that the split was done correctly. 
print()
print("Failure in training set before SMOTE:",y_train.sum()) #Prints the sum of the values in the y_train Series, which represents the number of instances of machine failure in the training set before applying SMOTE.
print("Non-failures in training set before SMOTE:", (y_train == 0).sum()) #Prints the count of instances in the y_train Series where the value is 0, which represents the number of non-failure instances in the training set before applying SMOTE.

#apply SMOTE to fix the imbalance in the training data
smote = SMOTE(random_state=42) #Creates an instance of the SMOTE class with a specified random state for reproducibility.in this case 42 is a common choice for random state, but it can be any integer.
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train) #Applies the fit_resample method of the SMOTE instance to the training feature set x_train and target variable y_train. This method generates synthetic samples for the minority class (machine failure) to balance the class distribution in the training data. The resulting resampled feature set is stored in x_train_resampled, and the corresponding target variable is stored in y_train_resampled.

print()
print("Failure in training after SMOTE:", y_train_resampled.sum()) #Prints the sum of the values in the y_train_resampled Series, which represents the number of instances of machine failure in the training set after applying SMOTE. This should show an increased number of failures compared to before SMOTE, indicating that the class imbalance has been addressed.
print("Non-failures in training after SMOTE:", (y_train_resampled == 0).sum()) #Prints the count of instances in the y_train_resampled Series where the value is 0, which represents the number of non-failure instances in the training set after applying SMOTE. This should show the same number of non-failures as before SMOTE, confirming that only the minority class (failures) has been oversampled to balance the dataset.

Training set size: (8000, 6)
Testing set size: (2000, 6)

Failure in training set before SMOTE: 271
Non-failures in training set before SMOTE: 7729

Failure in training after SMOTE: 7729
Non-failures in training after SMOTE: 7729


SMOTE created 7,458 synthetic failure examples by looking at the 271 real ones and generating similar but slightly different versions.
the model will now actually learn what a failure looks like.

train_test_split divides the data into two parts. 80% goes to training, meaning the model learns from it. 20% goes to testing, meaning i use it later to check how well the model performs on data it has never seen. 

stratify=y ensures both splits have the same proportion of failures so the test set is representative.

SMOTE stands for Synthetic Minority Oversampling Technique. It looks at the existing failure examples and generates new synthetic ones that are similar but not identical. 

This balances the training set so the model sees roughly equal amounts of failures and non-failures during training. Crucially i only apply SMOTE to the training set, never the test set, because the test set must reflect reality.

Step 6:
Train the first model

In [8]:
from sklearn.ensemble import RandomForestClassifier #Importing the RandomForestClassifier class from the sklearn.ensemble module, which is used to create a random forest model for classification tasks.
from sklearn.metrics import classification_report, confusion_matrix #Importing the classification_report and confusion_matrix functions from the sklearn.metrics module, which are used to evaluate the performance of the classification model by providing a detailed report of precision, recall, f1-score, and support for each class, as well as a confusion matrix that shows the counts of true positives, true negatives, false positives, and false negatives.

#train a random forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42) #Creates an instance of the RandomForestClassifier class with 100 trees (n_estimators=100) and a specified random state for reproducibility (random_state=42).
rf_model.fit(x_train_resampled, y_train_resampled) #Fits the random forest model rf_model to the resampled training data x_train_resampled and y_train_resampled. This process involves building multiple decision trees based on the training data and aggregating their predictions to create a robust classification model.

#evaluate the model on the test set
y_pred_rf = rf_model.predict(x_test) #Uses the predict method of the trained random forest model rf_model to generate predictions for the test feature set x_test. The predicted class labels are stored in y_pred_rf.

print("=== Random Forest Results ===") #Prints a header to indicate that the following output pertains to the results of the random forest model evaluation.
print()
print(classification_report(y_test, y_pred_rf)) #Prints the classification report for the random forest model by comparing the true labels y_test with the predicted labels y_pred_rf. The report includes precision, recall, f1-score, and support for each class, providing a detailed evaluation of the model's performance.
print()
print("Confusion Matrix:") #Prints a header to indicate that the following output is the confusion matrix for the random forest model evaluation.
print(confusion_matrix(y_test, y_pred_rf)) #Prints the confusion matrix for the random forest model by comparing the true labels y_test with the predicted labels y_pred_rf. The confusion matrix shows the counts of true positives, true negatives, false positives, and false negatives, which helps to understand the types of errors the model is making in its predictions.

=== Random Forest Results ===

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1932
           1       0.43      0.72      0.54        68

    accuracy                           0.96      2000
   macro avg       0.71      0.84      0.76      2000
weighted avg       0.97      0.96      0.96      2000


Confusion Matrix:
[[1866   66]
 [  19   49]]


What this does:
RandomForestClassifier builds 100 decision trees (n_estimators=100) and combines their votes to make predictions. It is one of the most reliable ML algorithms for this type of problem.

.fit() is where the actual learning happens. i feed it my balanced training data and it learns the patterns.

classification_report shows precision, recall, and F1 score for both classes. Remember recall is the most important metric here because missing a real failure is far worse than a false alarm.

confusion_matrix shows exactly how many predictions were correct and incorrect broken down by class.

Read it like this:

1866 : correctly predicted no failure (true negatives) ✅
66 : predicted no failure but it actually failed (false negatives) ❌ dangerous
19 : predicted failure but it was fine (false positives) ⚠️ minor inconvenience
49 : correctly predicted failure (true positives) ✅

So out of 68 real failures in the test set, the model caught 49 and missed 19.

The metrics that matter:
For class 1 (failures):

Precision 0.43 — when the model says failure, it is right 43% of the time
Recall 0.72 — the model catches 72% of all real failures
F1 0.54 — balance between the two

Recall is the key metric here. In an operations context missing a real failure costs thousands of pounds. A false alarm just means sending someone to check a machine unnecessarily. 72% recall on the first model with no tuning is actually decent.

The honest assessment:
Good: 72% recall, overall 96% accuracy, model is working
Needs improvement: still missing 28% of real failures and precision is low at 43%

This is completely normal for a first untuned model. The next two models (Logistic Regression and XGBoost) will give something to compare against, and XGBoost in particular should improve these numbers significantly.

Step 7
second model

In [ ]:
from sklearn.linear_model import LogisticRegression #Importing the LogisticRegression class from the sklearn.linear_model module, which is used to create a logistic regression model for classification tasks.
from sklearn.preprocessing import StandardScaler #Importing the StandardScaler class from the sklearn.preprocessing module, which is used to standardize features by removing the mean and scaling to unit variance. This is important for algorithms like logistic regression that are sensitive to the scale of the input features.
from sklearn.metrics import classification_report, confusion_matrix #Importing the classification_report and confusion_matrix functions from the sklearn.metrics module, which are used to evaluate the performance of the classification model by providing a detailed report of precision, recall, f1-score, and support for each class, as well as a confusion matrix that shows the counts of true positives, true negatives, false positives, and false negatives.

# Scale the features
scaler = StandardScaler() #Creates an instance of the StandardScaler class, which will be used to standardize the features in the training and testing datasets. Standardization is important for algorithms like logistic regression that are sensitive to the scale of the input features, as it helps to improve the convergence and performance of the model by ensuring that all features are on a similar scale.
x_train_scaled = scaler.fit_transform(x_train_resampled) #Applies the fit_transform method of the StandardScaler instance to the resampled training feature set x_train_resampled. This method computes the mean and standard deviation of each feature in the training data, then scales the features by removing the mean and scaling to unit variance. The resulting standardized training feature set is stored in x_train_scaled.
x_test_scaled = scaler.transform(x_test) #Applies the transform method of the StandardScaler instance to the testing feature set x_test. This method uses the mean and standard deviation computed from the training data to standardize the features in the testing set, ensuring consistency in scaling.

# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000) #Creates an instance of the LogisticRegression class with a specified random state for reproducibility (random_state=42) and a maximum number of iterations set to 1000 (max_iter=1000) to ensure that the optimization algorithm has enough iterations to converge, especially when dealing with larger datasets or more complex feature spaces.
lr_model.fit(x_train_scaled, y_train_resampled) #Fits the logistic regression model lr_model to the standardized resampled training data x_train_scaled and y_train_resampled. This process involves finding the optimal coefficients for the features in order to best separate the classes in the target variable, which in this case is whether the machine fails or not.

# Evaluate
y_pred_lr = lr_model.predict(x_test_scaled) #Uses the predict method of the trained logistic regression model lr_model to generate predictions for the standardized test feature set x_test_scaled. The predicted class labels are stored in y_pred_lr.

print("=== Logistic Regression Results ===") #Prints a header to indicate that the following output pertains to the results of the logistic regression model evaluation.
print()
print(classification_report(y_test, y_pred_lr)) #Prints the classification report for the logistic regression model by comparing the true labels y_test with the predicted labels y_pred_lr. The report includes precision, recall, f1-score, and support for each class, providing a detailed evaluation of the model's performance.
print()
print("Confusion Matrix:") #Prints a header to indicate that the following output is the confusion matrix for the logistic regression model evaluation.
print(confusion_matrix(y_test, y_pred_lr)) #Prints the confusion matrix for the logistic regression model by comparing the true labels y_test with the predicted labels y_pred_lr. The confusion matrix shows the counts of true positives, true negatives, false positives, and false negatives, which helps to understand the types of errors the model is making in its predictions.


=== Logistic Regression Results ===

              precision    recall  f1-score   support

           0       0.99      0.84      0.91      1932
           1       0.15      0.81      0.26        68

    accuracy                           0.84      2000
   macro avg       0.57      0.82      0.58      2000
weighted avg       0.96      0.84      0.89      2000


Confusion Matrix:
[[1625  307]
 [  13   55]]


StandardScaler rescales all the features so they are on the same scale. Logistic Regression is sensitive to features having very different ranges. For example rotational speed goes up to 2,886 while tool wear only goes to 253. Without scaling the model gives too much weight to the larger numbers. Random Forest does not need this but Logistic Regression does.

max_iter=1000 just gives the model enough iterations to find a solution. The default is sometimes too low for this type of data.

Logistic Regression has higher recall (0.81 vs 0.72) meaning it catches more real failures. It found 55 out of 68 failures compared to Random Forest finding 49. Good.

But Logistic Regression has much lower precision (0.15 vs 0.43) meaning it raises 307 false alarms compared to Random Forest's 66. In a real operations context that means sending engineers to check 307 machines unnecessarily. That gets expensive and annoying fast.

Random Forest gives a much better balance between catching failures and not crying wolf constantly. The F1 score of 0.54 vs 0.26 confirms this.

Neither model is perfect yet. That is exactly why i'm running XGBoost next; it typically outperforms both on this type of imbalanced industrial data.